In [0]:
# Gold Layer 1 - Admission Journey

spark.sql("CREATE CATALOG IF NOT EXISTS cdr_gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS cdr_gold.analytics_mimic_iv")

silver_catalog = "cdr_silver"
silver_schema = "processed_mimic_iv"

required_tables = [
    "admissions",
    "patients",
    "transfers",
    "services",
    "icustays",
    "icd_diagnoses",
    "d_icd_diagnoses",
    "icd_procedures",
]

for table in required_tables:
    full_name = f"{silver_catalog}.{silver_schema}.{table}"
    exists = spark.catalog.tableExists(full_name)
    print(f"{table:<20} {'FOUND' if exists else 'MISSING'}")

In [0]:
admissions_df = spark.table("cdr_silver.processed_mimic_iv.admissions")
patients_df = spark.table("cdr_silver.processed_mimic_iv.patients")

print("ADMISSIONS SCHEMA")
admissions_df.printSchema()

print("\nPATIENTS SCHEMA")
patients_df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Check expected uniqueness before joining
admission_rows = admissions_df.count()
admission_ids = admissions_df.select("hadm_id").distinct().count()

patient_rows = patients_df.count()
patient_ids = patients_df.select("subject_id").distinct().count()

print(f"Admissions: {admission_rows:,} rows | {admission_ids:,} unique hadm_id")
print(f"Patients:   {patient_rows:,} rows | {patient_ids:,} unique subject_id")

# Build admission-level base
base_df = (
    admissions_df.alias("a")
    .join(
        patients_df.alias("p"),
        F.col("a.subject_id") == F.col("p.subject_id"),
        "left"
    )
    .select(
        F.col("a.subject_id"),
        F.col("a.hadm_id"),

        F.col("p.gender"),
        F.col("p.anchor_age"),
        F.col("p.anchor_year"),
        F.col("p.anchor_year_group"),

        F.col("a.admittime"),
        F.col("a.dischtime"),
        F.col("a.deathtime"),
        F.col("a.admission_type"),
        F.col("a.admission_location"),
        F.col("a.discharge_location"),

        F.col("a.hospital_expire_flag").alias("mortality_flag"),

        F.to_date("a.admittime").alias("admission_date"),
        F.year("a.admittime").alias("admission_year"),
        F.month("a.admittime").alias("admission_month"),

        (
            F.col("p.anchor_age")
            + F.year("a.admittime")
            - F.col("p.anchor_year")
        ).alias("age_at_admission"),

        F.round(
            (
                F.col("a.dischtime").cast("long")
                - F.col("a.admittime").cast("long")
            ) / 86400.0,
            2
        ).alias("hospital_los_days")
    )
)

# Validate that the join kept one row per admission
base_rows = base_df.count()
base_hadm = base_df.select("hadm_id").distinct().count()

print(f"\nGold base:  {base_rows:,} rows | {base_hadm:,} unique hadm_id")
print(f"Row change after patient join: {base_rows - admission_rows:,}")

In [0]:
# Check age and LOS quality
base_df.select(
    F.count("*").alias("rows"),

    F.sum(F.col("age_at_admission").isNull().cast("int")).alias("age_nulls"),
    F.min("age_at_admission").alias("age_min"),
    F.max("age_at_admission").alias("age_max"),

    F.sum(F.col("hospital_los_days").isNull().cast("int")).alias("los_nulls"),
    F.min("hospital_los_days").alias("los_min"),
    F.max("hospital_los_days").alias("los_max"),

    F.sum((F.col("hospital_los_days") < 0).cast("int")).alias("negative_los")
).show(truncate=False)

# Distribution checkpoints
base_df.approxQuantile(
    ["age_at_admission", "hospital_los_days"],
    [0.01, 0.25, 0.50, 0.75, 0.95, 0.99],
    0.001
)

In [0]:
base_df = (
    base_df
    .withColumn(
        "age_group",
        F.when(F.col("age_at_admission") < 40, "18-39")
         .when(F.col("age_at_admission") < 60, "40-59")
         .when(F.col("age_at_admission") < 80, "60-79")
         .otherwise("80+")
    )
    .withColumn(
        "los_band",
        F.when(F.col("hospital_los_days") < 1, "<1 day")
         .when(F.col("hospital_los_days") < 3, "1-3 days")
         .when(F.col("hospital_los_days") < 7, "3-7 days")
         .when(F.col("hospital_los_days") < 14, "7-14 days")
         .otherwise("14+ days")
    )
)

In [0]:
base_df.groupBy("age_group").count().orderBy("age_group").show()

base_df.groupBy("los_band").count().orderBy(
    F.when(F.col("los_band") == "<1 day", 1)
     .when(F.col("los_band") == "1-3 days", 2)
     .when(F.col("los_band") == "3-7 days", 3)
     .when(F.col("los_band") == "7-14 days", 4)
     .otherwise(5)
).show()


In [0]:
transfers_df = spark.table(
    "cdr_silver.processed_mimic_iv.transfers"
)

print("TRANSFERS SCHEMA")
transfers_df.printSchema()

print("\nTRANSFER SUMMARY")

transfers_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("transfer_id").alias("unique_transfer_ids"),
    F.countDistinct("hadm_id").alias("admissions_with_transfers"),
    F.sum(F.col("hadm_id").isNull().cast("int")).alias("null_hadm_id")
).show(truncate=False)

In [0]:
admission_transfers_df = transfers_df.filter(
    F.col("hadm_id").isNotNull()
)

# Check transfer event types
admission_transfers_df.groupBy("eventtype").agg(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("admissions"),
    F.sum(F.col("careunit").isNull().cast("int")).alias("null_careunit")
).orderBy(F.desc("rows")).show(truncate=False)

# Check key journey fields
admission_transfers_df.select(
    F.count("*").alias("rows"),
    F.sum(F.col("careunit").isNull().cast("int")).alias("null_careunit"),
    F.sum(F.col("intime").isNull().cast("int")).alias("null_intime"),
    F.sum(F.col("outtime").isNull().cast("int")).alias("null_outtime")
).show(truncate=False)

In [0]:
transfers_df.select("eventtype").distinct().show()

In [0]:
from pyspark.sql.window import Window

# Keep only transfer records linked to a hospital admission
journey_events_df = transfers_df.filter(
    F.col("hadm_id").isNotNull()
)

first_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("intime").asc(),
        F.col("transfer_id").asc()
    )
)

last_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("intime").desc(),
        F.col("transfer_id").desc()
    )
)

# First care unit during the admission
first_units_df = (
    journey_events_df
    .withColumn("_rn", F.row_number().over(first_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("careunit").alias("first_careunit")
    )
)

# Last care unit during the admission
last_units_df = (
    journey_events_df
    .withColumn("_rn", F.row_number().over(last_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("careunit").alias("last_careunit")
    )
)

# Admission-level journey metrics
transfer_metrics_df = (
    journey_events_df
    .groupBy("hadm_id")
    .agg(
        F.sum(
            F.when(F.col("eventtype") == "transfer", 1).otherwise(0)
        ).alias("transfer_count"),

        F.countDistinct("careunit").alias("distinct_careunit_count")
    )
)

transfer_summary_df = (
    transfer_metrics_df
    .join(first_units_df, "hadm_id", "left")
    .join(last_units_df, "hadm_id", "left")
)

In [0]:
transfer_summary_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.min("transfer_count").alias("min_transfers"),
    F.max("transfer_count").alias("max_transfers"),
    F.min("distinct_careunit_count").alias("min_careunits"),
    F.max("distinct_careunit_count").alias("max_careunits")
).show(truncate=False)

In [0]:
gold_df = (
    base_df
    .join(
        transfer_summary_df,
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "transfer_count",
        F.coalesce(F.col("transfer_count"), F.lit(0))
    )
    .withColumn(
        "distinct_careunit_count",
        F.coalesce(F.col("distinct_careunit_count"), F.lit(0))
    )
    .withColumn(
        "first_careunit",
        F.coalesce(F.col("first_careunit"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "last_careunit",
        F.coalesce(F.col("last_careunit"), F.lit("UNKNOWN"))
    )
)

In [0]:
gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum((F.col("first_careunit") == "UNKNOWN").cast("int")).alias("unknown_first_careunit"),
    F.sum((F.col("last_careunit") == "UNKNOWN").cast("int")).alias("unknown_last_careunit")
).show(truncate=False)

In [0]:
(
    journey_events_df
    .groupBy("eventtype")
    .agg(
        F.count("*").alias("rows"),
        F.sum(
            (F.col("careunit") == "UNKNOWN").cast("int")
        ).alias("unknown_careunit")
    )
    .orderBy("eventtype")
    .show(truncate=False)
)

In [0]:
# Use only meaningful care units for first/last unit
known_unit_events_df = journey_events_df.filter(
    F.col("careunit") != "UNKNOWN"
)

first_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("intime").asc(),
        F.col("transfer_id").asc()
    )
)

last_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("intime").desc(),
        F.col("transfer_id").desc()
    )
)

first_units_df = (
    known_unit_events_df
    .withColumn("_rn", F.row_number().over(first_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("careunit").alias("first_careunit")
    )
)

last_units_df = (
    known_unit_events_df
    .withColumn("_rn", F.row_number().over(last_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("careunit").alias("last_careunit")
    )
)

transfer_summary_df = (
    transfer_metrics_df
    .join(first_units_df, "hadm_id", "left")
    .join(last_units_df, "hadm_id", "left")
)

In [0]:
gold_df = (
    base_df
    .join(
        transfer_summary_df,
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "transfer_count",
        F.coalesce(F.col("transfer_count"), F.lit(0))
    )
    .withColumn(
        "distinct_careunit_count",
        F.coalesce(F.col("distinct_careunit_count"), F.lit(0))
    )
    .withColumn(
        "first_careunit",
        F.coalesce(F.col("first_careunit"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "last_careunit",
        F.coalesce(F.col("last_careunit"), F.lit("UNKNOWN"))
    )
)

gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum((F.col("first_careunit") == "UNKNOWN").cast("int")).alias("unknown_first_careunit"),
    F.sum((F.col("last_careunit") == "UNKNOWN").cast("int")).alias("unknown_last_careunit")
).show(truncate=False)

In [0]:
services_df = spark.table(
    "cdr_silver.processed_mimic_iv.services"
)

print("SERVICES SCHEMA")
services_df.printSchema()

print("\nSERVICE SUMMARY")

services_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("admissions_with_services"),
    F.sum(F.col("hadm_id").isNull().cast("int")).alias("null_hadm_id"),
    F.sum(F.col("transfertime").isNull().cast("int")).alias("null_transfertime"),
    F.sum(F.col("curr_service").isNull().cast("int")).alias("null_curr_service"),
    F.sum(F.col("prev_service").isNull().cast("int")).alias("null_prev_service")
).show(truncate=False)

In [0]:
# Check service history structure
service_order_check = (
    services_df
    .groupBy("hadm_id", "transfertime")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Admission/time combinations with multiple service rows:",
    service_order_check.count()
)

services_df.select(
    F.count("*").alias("rows"),
    F.sum(F.col("prev_service").isNotNull().cast("int")).alias("rows_with_prev_service"),
    F.sum(
        (
            F.col("prev_service").isNotNull()
            & (F.col("prev_service") != F.col("curr_service"))
        ).cast("int")
    ).alias("actual_service_changes")
).show(truncate=False)

# How many service records does each admission have?
(
    services_df
    .groupBy("hadm_id")
    .count()
    .groupBy("count")
    .count()
    .orderBy("count")
    .show(20, truncate=False)
)

In [0]:
duplicate_service_times = (
    services_df
    .groupBy("hadm_id", "transfertime")
    .count()
    .filter(F.col("count") > 1)
)

(
    services_df.alias("s")
    .join(
        duplicate_service_times.alias("d"),
        (F.col("s.hadm_id") == F.col("d.hadm_id")) &
        (F.col("s.transfertime") == F.col("d.transfertime")),
        "inner"
    )
    .select(
        F.col("s.subject_id"),
        F.col("s.hadm_id"),
        F.col("s.transfertime"),
        F.col("s.prev_service"),
        F.col("s.curr_service")
    )
    .orderBy("hadm_id", "transfertime")
    .show(truncate=False)
)

In [0]:
(
    services_df
    .filter(F.col("hadm_id") == 21555795)
    .select(
        "subject_id",
        "hadm_id",
        "transfertime",
        "prev_service",
        "curr_service"
    )
    .orderBy("transfertime")
    .show(truncate=False)
)

In [0]:
# Order service history within each admission
first_service_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(F.col("transfertime").asc())
)

last_service_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(F.col("transfertime").desc())
)

# Initial service
initial_service_df = (
    services_df
    .withColumn("_rn", F.row_number().over(first_service_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("curr_service").alias("initial_service")
    )
)

# Final service
final_service_df = (
    services_df
    .withColumn("_rn", F.row_number().over(last_service_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("curr_service").alias("final_service")
    )
)

# Count genuine service changes
service_metrics_df = (
    services_df
    .groupBy("hadm_id")
    .agg(
        F.sum(
            F.when(
                F.col("prev_service").isNotNull()
                & (F.col("prev_service") != F.col("curr_service")),
                1
            ).otherwise(0)
        ).alias("service_change_count")
    )
)

service_summary_df = (
    service_metrics_df
    .join(initial_service_df, "hadm_id", "left")
    .join(final_service_df, "hadm_id", "left")
)

In [0]:
service_summary_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.min("service_change_count").alias("min_changes"),
    F.max("service_change_count").alias("max_changes")
).show(truncate=False)

In [0]:
gold_df = (
    gold_df
    .join(
        service_summary_df,
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "initial_service",
        F.coalesce(F.col("initial_service"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "final_service",
        F.coalesce(F.col("final_service"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "service_change_count",
        F.coalesce(F.col("service_change_count"), F.lit(0))
    )
    .withColumn(
        "security_service",
        F.col("initial_service")
    )
)

In [0]:
gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum((F.col("initial_service") == "UNKNOWN").cast("int")).alias("unknown_initial_service"),
    F.sum((F.col("final_service") == "UNKNOWN").cast("int")).alias("unknown_final_service"),
    F.min("service_change_count").alias("min_service_changes"),
    F.max("service_change_count").alias("max_service_changes")
).show(truncate=False)

In [0]:
icustays_df = spark.table(
    "cdr_silver.processed_mimic_iv.icustays"
)

print("ICUSTAYS SCHEMA")
icustays_df.printSchema()

print("\nICU SUMMARY")

icustays_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("stay_id").alias("unique_stay_id"),
    F.countDistinct("hadm_id").alias("admissions_with_icu"),
    F.sum(F.col("hadm_id").isNull().cast("int")).alias("null_hadm_id")
).show(truncate=False)

In [0]:
# Check ICU LOS quality
icustays_df.select(
    F.count("*").alias("rows"),
    F.sum(F.col("los").isNull().cast("int")).alias("null_los"),
    F.sum((F.col("los") < 0).cast("int")).alias("negative_los"),
    F.min("los").alias("min_los"),
    F.max("los").alias("max_los")
).show(truncate=False)

# ICU stays per hospital admission
(
    icustays_df
    .groupBy("hadm_id")
    .agg(F.countDistinct("stay_id").alias("icu_stay_count"))
    .groupBy("icu_stay_count")
    .count()
    .orderBy("icu_stay_count")
    .show(20, truncate=False)
)

In [0]:
(
    icustays_df
    .filter(F.col("los").isNull())
    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        "intime",
        "outtime",
        "los",
        F.round(
            (
                F.col("outtime").cast("long")
                - F.col("intime").cast("long")
            ) / 86400.0,
            5
        ).alias("calculated_los")
    )
    .show(20, truncate=False)
)

In [0]:
null_los_hadm = (
    icustays_df
    .filter(F.col("los").isNull())
    .select("hadm_id")
    .distinct()
)

(
    icustays_df
    .join(null_los_hadm, "hadm_id", "inner")
    .groupBy("hadm_id")
    .agg(
        F.countDistinct("stay_id").alias("icu_stay_count"),
        F.sum(F.col("los").isNull().cast("int")).alias("null_los_stays"),
        F.sum(F.col("los").isNotNull().cast("int")).alias("known_los_stays"),
        F.round(F.sum("los"), 2).alias("known_total_icu_los")
    )
    .orderBy("hadm_id")
    .show(20, truncate=False)
)

In [0]:
icu_summary_df = (
    icustays_df
    .groupBy("hadm_id")
    .agg(
        F.countDistinct("stay_id").alias("icu_stay_count"),

        F.sum(
            F.col("los").isNull().cast("int")
        ).alias("_missing_los_stays"),

        F.sum("los").alias("_known_icu_los_days")
    )
    .withColumn(
        "icu_flag",
        F.lit(1)
    )
    .withColumn(
        "icu_los_complete",
        F.col("_missing_los_stays") == 0
    )
    .withColumn(
        "total_icu_los_days",
        F.when(
            F.col("_missing_los_stays") == 0,
            F.round(F.col("_known_icu_los_days"), 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .drop("_missing_los_stays", "_known_icu_los_days")
)

In [0]:
icu_summary_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum((~F.col("icu_los_complete")).cast("int")).alias("incomplete_icu_los"),
    F.min("icu_stay_count").alias("min_stays"),
    F.max("icu_stay_count").alias("max_stays")
).show(truncate=False)

In [0]:
gold_df = (
    gold_df
    .join(
        icu_summary_df,
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "icu_flag",
        F.coalesce(F.col("icu_flag"), F.lit(0))
    )
    .withColumn(
        "icu_stay_count",
        F.coalesce(F.col("icu_stay_count"), F.lit(0))
    )
    .withColumn(
        "total_icu_los_days",
        F.when(
            F.col("icu_flag") == 0,
            F.lit(0.0)
        ).otherwise(F.col("total_icu_los_days"))
    )
    .withColumn(
        "icu_los_complete",
        F.when(
            F.col("icu_flag") == 0,
            F.lit(True)
        ).otherwise(F.col("icu_los_complete"))
    )
)


In [0]:
gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum(F.col("icu_flag")).alias("icu_admissions"),
    F.sum((F.col("icu_flag") == 0).cast("int")).alias("non_icu_admissions"),
    F.sum((~F.col("icu_los_complete")).cast("int")).alias("incomplete_icu_los")
).show(truncate=False)


In [0]:
diagnoses_df = spark.table(
    "cdr_silver.processed_mimic_iv.icd_diagnoses"
)

diagnosis_lookup_df = spark.table(
    "cdr_silver.processed_mimic_iv.d_icd_diagnoses"
)

print("ICD DIAGNOSES SCHEMA")
diagnoses_df.printSchema()

print("\nDIAGNOSIS LOOKUP SCHEMA")
diagnosis_lookup_df.printSchema()

print("\nDIAGNOSIS SUMMARY")

diagnoses_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("admissions_with_diagnosis"),
    F.sum(F.col("hadm_id").isNull().cast("int")).alias("null_hadm_id"),
    F.sum(F.col("seq_num").isNull().cast("int")).alias("null_seq_num")
).show(truncate=False)

# Check how seq_num behaves
(
    diagnoses_df
    .groupBy("seq_num")
    .count()
    .orderBy("seq_num")
    .show(15, truncate=False)
)

In [0]:
primary_diag_df = diagnoses_df.filter(F.col("seq_num") == 1)

# Admissions with more than one seq_num = 1 row
duplicate_primary_df = (
    primary_diag_df
    .groupBy("hadm_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Admissions with multiple seq_num = 1 diagnoses:",
    duplicate_primary_df.count()
)

# Show the actual duplicate-primary records
(
    primary_diag_df.alias("d")
    .join(
        duplicate_primary_df.alias("x"),
        F.col("d.hadm_id") == F.col("x.hadm_id"),
        "inner"
    )
    .select(
        F.col("d.subject_id"),
        F.col("d.hadm_id"),
        F.col("d.seq_num"),
        F.col("d.icd_code"),
        F.col("d.icd_version")
    )
    .orderBy("hadm_id", "icd_version", "icd_code")
    .show(30, truncate=False)
)


In [0]:
diagnosis_with_desc_df = (
    diagnoses_df.alias("d")
    .join(
        diagnosis_lookup_df.alias("l"),
        (F.col("d.icd_code") == F.col("l.icd_code")) &
        (F.col("d.icd_version") == F.col("l.icd_version")),
        "left"
    )
)

diagnosis_with_desc_df.select(
    F.count("*").alias("rows"),
    F.sum(F.col("l.description").isNull().cast("int")).alias("missing_descriptions")
).show(truncate=False)

In [0]:
# Join diagnosis descriptions
diagnosis_enriched_df = (
    diagnoses_df.alias("d")
    .join(
        diagnosis_lookup_df.alias("l"),
        (F.col("d.icd_code") == F.col("l.icd_code")) &
        (F.col("d.icd_version") == F.col("l.icd_version")),
        "left"
    )
    .select(
        F.col("d.subject_id"),
        F.col("d.hadm_id"),
        F.col("d.seq_num"),
        F.col("d.icd_code"),
        F.col("d.icd_version"),
        F.col("l.description")
    )
)

# Prefer ICD-10 when multiple seq_num = 1 rows exist
primary_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("seq_num").asc(),
        F.col("icd_version").desc(),
        F.col("icd_code").asc()
    )
)

primary_diagnosis_df = (
    diagnosis_enriched_df
    .filter(F.col("seq_num") == 1)
    .withColumn("_rn", F.row_number().over(primary_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("icd_code").alias("primary_diagnosis_code"),
        F.col("icd_version").alias("primary_diagnosis_version"),
        F.col("description").alias("primary_diagnosis")
    )
)

diagnosis_count_df = (
    diagnosis_enriched_df
    .groupBy("hadm_id")
    .agg(
        F.count("*").alias("diagnosis_count")
    )
)

diagnosis_summary_df = (
    diagnosis_count_df
    .join(primary_diagnosis_df, "hadm_id", "left")
)

In [0]:
diagnosis_summary_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum(F.col("primary_diagnosis").isNull().cast("int")).alias("missing_primary_diagnosis"),
    F.min("diagnosis_count").alias("min_diagnoses"),
    F.max("diagnosis_count").alias("max_diagnoses")
).show(truncate=False)

In [0]:
gold_df = (
    gold_df
    .join(
        diagnosis_summary_df,
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "diagnosis_count",
        F.coalesce(F.col("diagnosis_count"), F.lit(0))
    )
    .withColumn(
        "primary_diagnosis_code",
        F.coalesce(F.col("primary_diagnosis_code"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "primary_diagnosis",
        F.coalesce(F.col("primary_diagnosis"), F.lit("UNKNOWN"))
    )
)

In [0]:
gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum((F.col("diagnosis_count") == 0).cast("int")).alias("admissions_without_diagnosis"),
    F.sum((F.col("primary_diagnosis") == "UNKNOWN").cast("int")).alias("unknown_primary_diagnosis"),
    F.min("diagnosis_count").alias("min_diagnoses"),
    F.max("diagnosis_count").alias("max_diagnoses")
).show(truncate=False)

In [0]:
procedures_df = spark.table(
    "cdr_silver.processed_mimic_iv.icd_procedures"
)

print("ICD PROCEDURES SCHEMA")
procedures_df.printSchema()

print("\nPROCEDURE SUMMARY")

procedures_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("admissions_with_procedures"),
    F.sum(F.col("hadm_id").isNull().cast("int")).alias("null_hadm_id"),
    F.sum(F.col("seq_num").isNull().cast("int")).alias("null_seq_num")
).show(truncate=False)

In [0]:
procedure_summary_df = (
    procedures_df
    .groupBy("hadm_id")
    .agg(
        F.count("*").alias("procedure_count")
    )
)

In [0]:
gold_df = (
    gold_df
    .join(
        procedure_summary_df,
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "procedure_count",
        F.coalesce(F.col("procedure_count"), F.lit(0))
    )
)

In [0]:
gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum((F.col("procedure_count") == 0).cast("int")).alias("admissions_without_procedures"),
    F.min("procedure_count").alias("min_procedures"),
    F.max("procedure_count").alias("max_procedures")
).show(truncate=False)

In [0]:
print("FINAL GOLD SCHEMA")
gold_df.printSchema()

print("\nFINAL QUALITY CHECK")

gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.sum(F.col("hadm_id").isNull().cast("int")).alias("null_hadm_id"),
    F.sum(F.col("subject_id").isNull().cast("int")).alias("null_subject_id"),

    F.sum((F.col("first_careunit") == "UNKNOWN").cast("int")).alias("unknown_first_careunit"),
    F.sum((F.col("initial_service") == "UNKNOWN").cast("int")).alias("unknown_initial_service"),

    F.sum(F.col("mortality_flag")).alias("in_hospital_deaths"),

    F.sum(F.col("icu_flag")).alias("icu_admissions"),
    F.sum((~F.col("icu_los_complete")).cast("int")).alias("incomplete_icu_los"),

    F.sum((F.col("diagnosis_count") == 0).cast("int")).alias("no_diagnosis"),
    F.sum((F.col("procedure_count") == 0).cast("int")).alias("no_procedure")
).show(truncate=False)

In [0]:
gold_df.select(
    F.sum(
        (
            (F.col("icu_flag") == 0) &
            (
                (F.col("icu_stay_count") != 0) |
                (F.col("total_icu_los_days") != 0)
            )
        ).cast("int")
    ).alias("invalid_non_icu_rows"),

    F.sum(
        (
            (F.col("icu_flag") == 1) &
            (F.col("icu_stay_count") == 0)
        ).cast("int")
    ).alias("invalid_icu_rows"),

    F.sum(
        (F.col("hospital_los_days") < 0).cast("int")
    ).alias("negative_hospital_los"),

    F.sum(
        (F.col("age_at_admission") < 18).cast("int")
    ).alias("age_under_18")
).show(truncate=False)

In [0]:
final_gold_df = (
    gold_df
    .select(
        # Keys
        "subject_id",
        "hadm_id",

        # Patient
        "gender",
        "age_at_admission",
        "age_group",

        # Admission
        "admittime",
        "dischtime",
        "admission_date",
        "admission_year",
        "admission_month",
        "admission_type",
        "admission_location",
        "discharge_location",

        # Outcomes
        "hospital_los_days",
        "los_band",
        "mortality_flag",

        # Hospital journey
        "first_careunit",
        "last_careunit",
        "transfer_count",
        "distinct_careunit_count",

        # Service
        "initial_service",
        "final_service",
        "service_change_count",
        "security_service",

        # ICU
        "icu_flag",
        "icu_stay_count",
        "total_icu_los_days",
        "icu_los_complete",

        # Clinical
        "primary_diagnosis_code",
        "primary_diagnosis_version",
        "primary_diagnosis",
        "diagnosis_count",
        "procedure_count"
    )
    .withColumn(
        "_gold_timestamp",
        F.current_timestamp()
    )
)

target_table = "cdr_gold.analytics_mimic_iv.admission_journey"

(
    final_gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Gold table created: {target_table}")

In [0]:
published_gold_df = spark.table(
    "cdr_gold.analytics_mimic_iv.admission_journey"
)

published_gold_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_id"),
    F.countDistinct("subject_id").alias("unique_patients"),
    F.sum("icu_flag").alias("icu_admissions"),
    F.sum("mortality_flag").alias("in_hospital_deaths")
).show(truncate=False)

In [0]:
spark.sql("""
COMMENT ON TABLE cdr_gold.analytics_mimic_iv.admission_journey IS
'Admission-level Gold dataset for MIMIC-IV. One row represents one hospital admission and summarises patient demographics, hospital journey, clinical service, ICU utilisation, primary diagnosis and procedure activity.'
""")

comments = {
    "subject_id": "Deidentified patient identifier.",
    "hadm_id": "Unique hospital admission identifier and grain of this Gold table.",
    "age_at_admission": "Derived patient age at the time of hospital admission.",
    "hospital_los_days": "Hospital length of stay in days from admission to discharge.",
    "mortality_flag": "Indicates whether the patient died during the hospital admission.",
    "first_careunit": "First known care unit recorded during the hospital admission.",
    "last_careunit": "Last known care unit recorded before discharge.",
    "transfer_count": "Number of recorded transfer events during the hospital admission.",
    "initial_service": "First clinical service responsible for the admission.",
    "final_service": "Final clinical service responsible for the admission.",
    "security_service": "Service field intended for service-based row-level access controls.",
    "icu_flag": "Indicates whether the hospital admission included at least one ICU stay.",
    "icu_stay_count": "Number of distinct ICU stays during the hospital admission.",
    "total_icu_los_days": "Total ICU length of stay in days. Null when one or more ICU stays have incomplete LOS.",
    "icu_los_complete": "Indicates whether ICU LOS is complete for all ICU stays in the admission.",
    "primary_diagnosis_code": "Primary ICD diagnosis code for the admission.",
    "primary_diagnosis_version": "ICD version associated with the primary diagnosis code.",
    "primary_diagnosis": "Description of the primary diagnosis.",
    "diagnosis_count": "Number of diagnosis records associated with the admission.",
    "procedure_count": "Number of ICD procedure records associated with the admission."
}

for column, description in comments.items():
    spark.sql(f"""
        ALTER TABLE cdr_gold.analytics_mimic_iv.admission_journey
        ALTER COLUMN {column}
        COMMENT '{description}'
    """)

print("Gold table metadata added.")